<a href="https://colab.research.google.com/github/kupalmananalsal-hub/Project_Pi/blob/main/notebooks/train_age_diverse_multilingual_kws.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Age-Diverse Multilingual openWakeWord Training

This Colab notebook prepares real and synthetic English/Tagalog distress-phrase audio, balances age-diverse positive clips, trains one openWakeWord ONNX model per phrase, evaluates held-out clips, and packages `.onnx` plus `.onnx.data` files for Raspberry Pi deployment.

Run the dependency cell first, restart the runtime when it asks, then continue from the import/config cell. The defaults use a Colab-friendly profile; switch `RUNTIME_PROFILE = "final"` before final training.


## Cell A - Install Dependencies

Run this cell by itself the first time. It intentionally restarts the Colab runtime after changing NumPy, Pandas, PyArrow, and SciPy. After Colab reconnects, continue from Cell A2 and do not rerun Cell A unless you start from a fresh runtime.


In [1]:
from pathlib import Path
import os
import signal
import subprocess
import sys

CONSTRAINTS_PATH = Path("project_pi_constraints.txt")
INSTALL_SENTINEL = Path("/content/project_pi_dependencies_installed.ok")

CONSTRAINTS_PATH.write_text("""\
numpy==1.26.4
pandas==2.2.2
pyarrow==15.0.2
datasets==2.19.2
scipy==1.13.1
soundfile>=0.12.1,<0.14
fsspec==2024.3.1
requests==2.32.4
""")


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


if INSTALL_SENTINEL.exists():
    print("Dependency install was already completed in this Colab runtime.")
    print("Continue from Cell A2. If imports still fail, delete the runtime and run Cell A again.")
else:
    pip_install(
        "--no-cache-dir",
        "--force-reinstall",
        "-c",
        str(CONSTRAINTS_PATH),
        "numpy==1.26.4",
        "pandas==2.2.2",
        "pyarrow==15.0.2",
        "datasets[audio]==2.19.2",
        "scipy==1.13.1",
        "soundfile>=0.12.1,<0.14",
        "fsspec==2024.3.1",
        "requests==2.32.4",
    )

    pip_install(
        "-c",
        str(CONSTRAINTS_PATH),
        "librosa==0.10.2.post1",
        "audiomentations==0.33.0",
        "edge-tts==6.1.12",
        "transformers==4.41.2",
        "accelerate==0.30.1",
        "torchaudio",
        "onnx",
        "onnxscript",
        "onnxruntime",
        "tqdm",
        "jiwer",
        "pyyaml",
    )

    INSTALL_SENTINEL.write_text("ok\n", encoding="utf-8")
    print("Dependency install complete. Restarting the Colab runtime now.")
    print("After Colab reconnects, continue from Cell A2 and do not rerun Cell A.")
    os.kill(os.getpid(), signal.SIGKILL)

Dependency install was already completed in this Colab runtime.
Continue from Cell A2. If imports still fail, delete the runtime and run Cell A again.


## Cell A2 - Imports And Configuration

Set `HF_TOKEN` in Colab secrets or `os.environ["HF_TOKEN"]` if you want gated Common Voice or optional private datasets. Public FLEURS and LibriSpeech work without a token.


In [2]:
import asyncio
import hashlib
import importlib
import inspect
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from math import gcd
from pathlib import Path

import datasets
import librosa
import numpy as np
import pandas as pd
import pyarrow as pa
import scipy
import soundfile as sf
import torch
from audiomentations import AddGaussianNoise, Compose, Gain, PitchShift, TimeStretch
from scipy.signal import resample_poly
from tqdm.auto import tqdm

try:
    from google.colab import userdata
except Exception:
    userdata = None

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN is None and userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SAMPLE_RATE = 16000
WORK_DIR = Path("/content/project_pi_kws")
REAL_CLIPS_DIR = WORK_DIR / "real_clips"
TTS_CLIPS_DIR = WORK_DIR / "tts_clips"
POSITIVE_CLIPS_DIR = WORK_DIR / "positive_clips"
HELDOUT_CLIPS_DIR = WORK_DIR / "heldout_clips"
AUGMENTATION_DIR = WORK_DIR / "augmentation"
OPENWAKEWORD_ROOT = WORK_DIR / "openwakeword"
PIPER_ROOT = WORK_DIR / "piper-sample-generator"
MODEL_ROOT = Path("/content/my_custom_model")

for directory in [
    WORK_DIR,
    REAL_CLIPS_DIR,
    TTS_CLIPS_DIR,
    POSITIVE_CLIPS_DIR,
    HELDOUT_CLIPS_DIR,
    AUGMENTATION_DIR,
    MODEL_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

RUNTIME_PROFILE = "colab_4h"  # "colab_4h" or "final"
FINAL_TRAINING = RUNTIME_PROFILE == "final"

MAX_ROWS_PER_STREAMED_SOURCE = 2500 if not FINAL_TRAINING else 12000
MAX_ASR_ROWS_PER_SOURCE = 600 if not FINAL_TRAINING else 3000
MAX_REAL_CLIPS_PER_KEYWORD = 120 if not FINAL_TRAINING else 600
MIN_POSITIVE_CLIPS_PER_KEYWORD = 500
HELDOUT_FRACTION = 0.15

TRAIN_KEYWORDS = None  # None trains all keywords; use ["help", "tulong"] for a quick focused run.
EVALUATION_MAX_POSITIVES = 200
EVALUATION_MAX_NEGATIVES = 400

ENGLISH_KEYWORDS = [
    "help",
    "help me",
    "save me",
    "please help",
    "emergency",
    "rescue",
    "over here",
    "ouch",
]

TAGALOG_KEYWORDS = [
    "tulong",
    "saklolo",
    "tulungan niyo ako",
    "tulungan mo ako",
    "ang sakit",
    "aray",
    "sunog",
    "agai",
    "kailangan ko ng tulong",
]

KEYWORD_SPECS = [
    {"language": "en", "phrase": phrase, "model_name": phrase.replace(" ", "_")}
    for phrase in ENGLISH_KEYWORDS
] + [
    {"language": "fil", "phrase": phrase, "model_name": phrase.replace(" ", "_")}
    for phrase in TAGALOG_KEYWORDS
]

if TRAIN_KEYWORDS:
    selected = {item.replace(" ", "_") for item in TRAIN_KEYWORDS}
    KEYWORD_SPECS = [item for item in KEYWORD_SPECS if item["model_name"] in selected]

MODEL_TO_PHRASE = {item["model_name"]: item["phrase"] for item in KEYWORD_SPECS}
MODEL_TO_LANGUAGE = {item["model_name"]: item["language"] for item in KEYWORD_SPECS}

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)
print("datasets:", datasets.__version__)
print("scipy:", scipy.__version__)
print("torch:", torch.__version__)
print(f"Configured {len(KEYWORD_SPECS)} keyword models")
print("HF token:", "present" if HF_TOKEN else "not set; gated sources will be skipped")

numpy: 1.26.4
pandas: 2.2.2
pyarrow: 15.0.2
datasets: 2.19.2
scipy: 1.13.1
torch: 2.11.0+cu128
Configured 17 keyword models
HF token: not set; gated sources will be skipped


## Cell B - Dataset Sources

The source registry lists exact HuggingFace dataset IDs and config/split names. Common Voice and the optional Filipino dataset can require authentication or terms acceptance, so the loader records a skip instead of stopping the notebook.


In [3]:
@dataclass(frozen=True)
class SpeechSource:
    dataset_id: str
    config: str | None
    split: str
    language: str
    text_fields: tuple[str, ...]
    age_field: str | None = None
    gender_field: str | None = None
    audio_field: str = "audio"
    streaming: bool = True
    requires_auth: bool = False
    note: str = ""


SPEECH_SOURCES = [
    SpeechSource(
        dataset_id="mozilla-foundation/common_voice_11_0",
        config="fil",
        split="train",
        language="fil",
        text_fields=("sentence", "text", "transcription"),
        age_field="age",
        gender_field="gender",
        streaming=True,
        requires_auth=True,
        note="Filipino Common Voice with age/gender when HF access is available.",
    ),
    SpeechSource(
        dataset_id="mozilla-foundation/common_voice_11_0",
        config="en",
        split="train",
        language="en",
        text_fields=("sentence", "text", "transcription"),
        age_field="age",
        gender_field="gender",
        streaming=True,
        requires_auth=True,
        note="English Common Voice with age/gender when HF access is available.",
    ),
    SpeechSource(
        dataset_id="google/fleurs",
        config="fil_ph",
        split="train",
        language="fil",
        text_fields=("transcription", "raw_transcription", "sentence", "text"),
        gender_field="gender",
        streaming=True,
        note="Public Filipino FLEURS split; no age metadata.",
    ),
    SpeechSource(
        dataset_id="google/fleurs",
        config="en_us",
        split="train",
        language="en",
        text_fields=("transcription", "raw_transcription", "sentence", "text"),
        gender_field="gender",
        streaming=True,
        note="Public US English FLEURS split; no age metadata.",
    ),
    SpeechSource(
        dataset_id="cdminix/filipino-voice-dataset",
        config=None,
        split="train",
        language="fil",
        text_fields=("sentence", "text", "transcription"),
        age_field="age",
        gender_field="gender",
        streaming=True,
        requires_auth=True,
        note="Optional Filipino voice dataset; skipped if not accessible.",
    ),
    SpeechSource(
        dataset_id="openslr/librispeech_asr",
        config="clean",
        split="train.100",
        language="en",
        text_fields=("text", "sentence", "transcription"),
        streaming=True,
        note="Adult read English speech from LibriSpeech.",
    ),
    SpeechSource(
        dataset_id="PolyAI/minds14",
        config="en-US",
        split="train",
        language="en",
        text_fields=("transcription", "text", "sentence"),
        streaming=True,
        note="Additional public English accents/intents; no age metadata.",
    ),
]

def source_label(source: SpeechSource) -> str:
    config = source.config or "default"
    return f"{source.dataset_id}:{config}:{source.split}"

print("Speech dataset registry:")
for source in SPEECH_SOURCES:
    auth = "requires HF_TOKEN" if source.requires_auth else "public"
    print(f"- {source_label(source)} [{source.language}, {auth}] {source.note}")

Speech dataset registry:
- mozilla-foundation/common_voice_11_0:fil:train [fil, requires HF_TOKEN] Filipino Common Voice with age/gender when HF access is available.
- mozilla-foundation/common_voice_11_0:en:train [en, requires HF_TOKEN] English Common Voice with age/gender when HF access is available.
- google/fleurs:fil_ph:train [fil, public] Public Filipino FLEURS split; no age metadata.
- google/fleurs:en_us:train [en, public] Public US English FLEURS split; no age metadata.
- cdminix/filipino-voice-dataset:default:train [fil, requires HF_TOKEN] Optional Filipino voice dataset; skipped if not accessible.
- openslr/librispeech_asr:clean:train.100 [en, public] Adult read English speech from LibriSpeech.
- PolyAI/minds14:en-US:train [en, public] Additional public English accents/intents; no age metadata.


In [4]:
def dataset_load_kwargs(source: SpeechSource) -> dict:
    kwargs = {
        "path": source.dataset_id,
        "name": source.config,
        "split": source.split,
        "streaming": source.streaming,
        "trust_remote_code": True,
    }
    if source.config is None:
        kwargs.pop("name")
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return kwargs


def load_speech_source(source: SpeechSource):
    try:
        dataset = datasets.load_dataset(**dataset_load_kwargs(source))
        dataset = dataset.cast_column(source.audio_field, datasets.Audio(sampling_rate=SAMPLE_RATE))
        print(f"Loaded {source_label(source)}")
        return dataset
    except Exception as exc:
        print(f"Skipping {source_label(source)}: {type(exc).__name__}: {exc}")
        return None


loaded_speech_sources = {}
for source in SPEECH_SOURCES:
    loaded_speech_sources[source_label(source)] = load_speech_source(source)

available_sources = [
    label for label, dataset in loaded_speech_sources.items()
    if dataset is not None
]
print(f"Available speech sources: {len(available_sources)}/{len(SPEECH_SOURCES)}")
for label in available_sources:
    print("  ", label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Skipping mozilla-foundation/common_voice_11_0:fil:train: DatasetNotFoundError: Dataset 'mozilla-foundation/common_voice_11_0' doesn't exist on the Hub or cannot be accessed. If the dataset is private or gated, make sure to log in with `huggingface-cli login` or visit the dataset page at https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0 to ask for access.
Skipping mozilla-foundation/common_voice_11_0:en:train: DatasetNotFoundError: Dataset 'mozilla-foundation/common_voice_11_0' doesn't exist on the Hub or cannot be accessed. If the dataset is private or gated, make sure to log in with `huggingface-cli login` or visit the dataset page at https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0 to ask for access.


Loaded google/fleurs:fil_ph:train
Loaded google/fleurs:en_us:train
Skipping cdminix/filipino-voice-dataset:default:train: DatasetNotFoundError: Dataset 'cdminix/filipino-voice-dataset' doesn't exist on the Hub or cannot be accessed. If the dataset is private or gated, make sure to log in with `huggingface-cli login` or visit the dataset page at https://huggingface.co/datasets/cdminix/filipino-voice-dataset to ask for access.


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loaded openslr/librispeech_asr:clean:train.100


Loaded PolyAI/minds14:en-US:train
Available speech sources: 4/7
   google/fleurs:fil_ph:train
   google/fleurs:en_us:train
   openslr/librispeech_asr:clean:train.100
   PolyAI/minds14:en-US:train


## Cell B2 - Augmentation Assets

MIT RIRs, AudioSet, FMA, and openWakeWord feature files are used for augmentation and false-positive suppression. AudioSet's current HuggingFace repository stores balanced data as Parquet, so this cell streams `agkphysics/AudioSet` instead of downloading the removed `bal_train09.tar` file. If an optional background source fails, the notebook keeps going and adds generated noise backgrounds.


In [ ]:
def _to_numpy(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def _to_mono(array):
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        if array.shape[0] <= 8 and array.shape[1] > array.shape[0]:
            array = array.mean(axis=0)
        else:
            array = array.mean(axis=1)
    return np.squeeze(array)


def decode_audio_value(audio):
    if isinstance(audio, dict):
        return _to_mono(audio["array"]), int(audio["sampling_rate"]), audio.get("path")
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return _to_mono(_to_numpy(samples.data)), int(samples.sample_rate), None
    raise TypeError(f"Unsupported audio value from datasets: {type(audio)!r}")


def write_wav_16k(output_path, audio, source_sample_rate=None):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(audio, dict) or hasattr(audio, "get_all_samples"):
        array, sample_rate, _ = decode_audio_value(audio)
    else:
        array = _to_mono(audio)
        sample_rate = int(source_sample_rate or SAMPLE_RATE)
    array = np.nan_to_num(np.asarray(array, dtype=np.float32))
    if sample_rate != SAMPLE_RATE:
        divisor = gcd(int(sample_rate), SAMPLE_RATE)
        array = resample_poly(array, SAMPLE_RATE // divisor, int(sample_rate) // divisor)
    if array.size == 0:
        raise ValueError(f"Decoded empty audio for {output_path}")
    peak = float(np.max(np.abs(array)))
    if peak > 1.0:
        array = array / peak
    sf.write(str(output_path), np.clip(array, -1.0, 1.0), SAMPLE_RATE, subtype="PCM_16")
    return output_path


def count_wavs(directory: Path) -> int:
    directory = Path(directory)
    if not directory.exists():
        return 0
    return sum(1 for _ in directory.glob("*.wav"))


def safe_filename(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(value or "")).strip("_")
    return value[:80] or "audio"


def create_synthetic_backgrounds(noise_dir: Path) -> int:
    noise_dir.mkdir(parents=True, exist_ok=True)
    target = 80 if not FINAL_TRAINING else 240
    existing = count_wavs(noise_dir)
    if existing >= target:
        return existing

    rng = np.random.default_rng(RANDOM_SEED)
    for index in tqdm(range(existing, target), desc="Synthetic background"):
        duration = float(rng.uniform(4.0, 10.0))
        n_samples = int(duration * SAMPLE_RATE)
        t = np.arange(n_samples, dtype=np.float32) / SAMPLE_RATE
        audio = rng.normal(0.0, float(rng.uniform(0.006, 0.035)), n_samples).astype(np.float32)

        hum_freq = float(rng.choice([50, 60, 100, 120, 180, 240]))
        audio += (0.006 * np.sin(2 * np.pi * hum_freq * t)).astype(np.float32)

        for _ in range(int(rng.integers(1, 5))):
            start = int(rng.integers(0, max(1, n_samples - SAMPLE_RATE // 4)))
            length = int(rng.integers(SAMPLE_RATE // 20, SAMPLE_RATE // 2))
            end = min(n_samples, start + length)
            audio[start:end] += rng.normal(0.0, float(rng.uniform(0.015, 0.06)), end - start)

        peak = float(np.max(np.abs(audio))) or 1.0
        audio = audio / max(1.0, peak * 1.2)
        sf.write(noise_dir / f"synthetic_noise_{index:04d}.wav", audio, SAMPLE_RATE, subtype="PCM_16")

    return count_wavs(noise_dir)


def download_mit_rirs(rir_dir: Path) -> int:
    rir_dir.mkdir(parents=True, exist_ok=True)
    if count_wavs(rir_dir):
        return count_wavs(rir_dir)
    try:
        rir_dataset = datasets.load_dataset(
            "davidscripka/MIT_environmental_impulse_responses",
            split="train",
            streaming=True,
            token=HF_TOKEN if HF_TOKEN else None,
        ).cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
        for index, row in enumerate(tqdm(rir_dataset, desc="MIT RIRs")):
            _, _, audio_path = decode_audio_value(row["audio"])
            name = Path(audio_path).name if audio_path else f"rir_{index:05d}.wav"
            write_wav_16k(rir_dir / name, row["audio"])
    except Exception as exc:
        print(f"Skipping MIT RIR download: {type(exc).__name__}: {exc}")
    return count_wavs(rir_dir)


def keep_audioset_background(row: dict) -> bool:
    labels = [str(label).lower() for label in row.get("human_labels", []) or []]
    blocked_terms = ("speech", "conversation", "narration", "monologue", "babbling")
    return not any(any(term in label for term in blocked_terms) for label in labels)


def build_audioset_background(audioset_16k: Path) -> int:
    audioset_16k.mkdir(parents=True, exist_ok=True)
    target = 600 if not FINAL_TRAINING else 3000
    existing = count_wavs(audioset_16k)
    if existing >= target:
        return existing

    try:
        audioset_dataset = datasets.load_dataset(
            "agkphysics/AudioSet",
            "balanced",
            split="train",
            streaming=True,
            token=HF_TOKEN if HF_TOKEN else None,
        ).cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))

        written = existing
        for row in tqdm(audioset_dataset, desc="AudioSet balanced background"):
            if not keep_audioset_background(row):
                continue
            _, _, audio_path = decode_audio_value(row["audio"])
            stem = Path(audio_path).stem if audio_path else safe_filename(row.get("video_id", written))
            output_path = audioset_16k / f"{safe_filename(stem)}.wav"
            if output_path.exists():
                continue
            write_wav_16k(output_path, row["audio"])
            written += 1
            if written >= target:
                break
        if written == existing:
            print("AudioSet streamed successfully but no non-speech background rows were saved.")
    except Exception as exc:
        print(f"Skipping AudioSet background stream: {type(exc).__name__}: {exc}")

    return count_wavs(audioset_16k)


def build_fma_background(fma_dir: Path) -> int:
    fma_dir.mkdir(parents=True, exist_ok=True)
    target = 120 if not FINAL_TRAINING else 360
    existing = count_wavs(fma_dir)
    if existing >= target:
        return existing

    try:
        fma_dataset = datasets.load_dataset(
            "rudraml/fma",
            name="small",
            split="train",
            streaming=True,
            token=HF_TOKEN if HF_TOKEN else None,
        )
        fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE)))
        for index in tqdm(range(existing, target), desc="FMA 16 kHz"):
            row = next(fma_dataset)
            _, _, audio_path = decode_audio_value(row["audio"])
            name = Path(audio_path).name.replace(".mp3", ".wav") if audio_path else f"fma_{index:05d}.wav"
            write_wav_16k(fma_dir / name, row["audio"])
    except Exception as exc:
        print(f"Skipping FMA background stream: {type(exc).__name__}: {exc}")

    return count_wavs(fma_dir)


def download_required_feature_files():
    feature_urls = {
        "openwakeword_features_ACAV100M_2000_hrs_16bit.npy": "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
        "validation_set_features.npy": "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy",
    }
    for filename, url in feature_urls.items():
        if Path(filename).exists():
            continue
        command = ["wget", "-nc", url, "-O", filename]
        if HF_TOKEN:
            command.insert(1, f"--header=Authorization: Bearer {HF_TOKEN}")
        subprocess.run(command, check=True)


def download_augmentation_assets():
    rir_dir = AUGMENTATION_DIR / "mit_rirs"
    audioset_16k = AUGMENTATION_DIR / "audioset_16k"
    fma_dir = AUGMENTATION_DIR / "fma"
    synthetic_noise_dir = AUGMENTATION_DIR / "synthetic_noise"

    counts = {
        "rir_wavs": download_mit_rirs(rir_dir),
        "audioset_wavs": build_audioset_background(audioset_16k),
        "fma_wavs": build_fma_background(fma_dir),
        "synthetic_noise_wavs": create_synthetic_backgrounds(synthetic_noise_dir),
    }
    download_required_feature_files()

    return {
        "rir_dir": str(rir_dir),
        "audioset_16k": str(audioset_16k),
        "fma": str(fma_dir),
        "synthetic_noise": str(synthetic_noise_dir),
        "counts": counts,
    }


augmentation_assets = download_augmentation_assets()
print(json.dumps(augmentation_assets, indent=2))

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

MIT RIRs: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

AudioSet balanced background: 0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1491: FutureWarning: The repository for rudraml/fma contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/rudraml/fma
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Skipping FMA background stream: ValueError: Cannot seek streaming HTTP file


Synthetic background:   0%|          | 0/80 [00:00<?, ?it/s]

## Cell C - Extract Real Keyword Clips

This cell uses dataset transcripts first when present, then ASR for bounded fallback scanning. Tagalog ASR uses `Khalsuu/filipino-wav2vec2-l-xls-r-300m-official`; English ASR uses `openai/whisper-tiny` by default.


In [ ]:
TAGALOG_ASR_MODEL = "Khalsuu/filipino-wav2vec2-l-xls-r-300m-official"
ENGLISH_ASR_MODEL = "openai/whisper-tiny"
USE_REFERENCE_TRANSCRIPTS_FIRST = True
RUN_ASR_ON_REFERENCE_MISMATCH = True


def slugify(value: str, fallback: str = "unknown") -> str:
    value = str(value or "").strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or fallback


def normalize_text(value: str) -> str:
    value = str(value or "").lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    return " ".join(value.split())


def phrase_present(text: str, phrase: str) -> bool:
    normalized_text = f" {normalize_text(text)} "
    normalized_phrase = normalize_text(phrase)
    return bool(normalized_phrase) and f" {normalized_phrase} " in normalized_text


def row_text(row: dict, source: SpeechSource) -> str:
    for field in source.text_fields:
        value = row.get(field)
        if value:
            return str(value)
    return ""


def normalize_age(row: dict, source: SpeechSource) -> str:
    value = str(row.get(source.age_field) or "unknown").strip().lower() if source.age_field else "unknown"
    aliases = {
        "teens": "teenager",
        "twenties": "adult",
        "thirties": "adult",
        "fourties": "adult",
        "forties": "adult",
        "fifties": "adult",
        "sixties": "older",
        "seventies": "older",
        "eighties": "older",
    }
    return slugify(aliases.get(value, value), "unknown")


def normalize_gender(row: dict, source: SpeechSource) -> str:
    if not source.gender_field:
        return "u"
    value = str(row.get(source.gender_field) or "u").strip().lower()
    if value in {"0", "male", "m"}:
        return "m"
    if value in {"1", "female", "f"}:
        return "f"
    return slugify(value, "u")[:12]


_ASR_PIPELINES = {}


def get_asr_pipeline(language: str):
    key = "fil" if language == "fil" else "en"
    if key not in _ASR_PIPELINES:
        from transformers import pipeline

        model_id = TAGALOG_ASR_MODEL if key == "fil" else ENGLISH_ASR_MODEL
        task_kwargs = {"language": "english", "task": "transcribe"} if key == "en" and "whisper" in model_id else {}
        _ASR_PIPELINES[key] = pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=0 if torch.cuda.is_available() else -1,
            model_kwargs={"low_cpu_mem_usage": True} if key == "en" else {},
            generate_kwargs=task_kwargs,
        )
    return _ASR_PIPELINES[key]


def transcribe_audio(audio, language: str) -> str:
    array, sample_rate, _ = decode_audio_value(audio)
    array = np.asarray(array, dtype=np.float32)
    recognizer = get_asr_pipeline(language)
    result = recognizer({"array": array, "sampling_rate": sample_rate})
    return result.get("text", "") if isinstance(result, dict) else str(result)


def save_real_keyword_clip(row: dict, source: SpeechSource, model_name: str, index: int) -> Path:
    age = normalize_age(row, source)
    gender = normalize_gender(row, source)
    dataset_slug = slugify(source.dataset_id.split("/")[-1])
    output_dir = REAL_CLIPS_DIR / source.language / model_name
    filename = f"{model_name}_{age}_{gender}_{dataset_slug}_{index:06d}.wav"
    return write_wav_16k(output_dir / filename, row[source.audio_field])


def extract_real_clips_from_source(source: SpeechSource, dataset) -> dict[str, int]:
    counts = Counter()
    asr_rows = 0
    scanned_rows = 0
    source_keywords = [
        item for item in KEYWORD_SPECS
        if item["language"] == source.language
    ]
    per_keyword_limit = {
        item["model_name"]: MAX_REAL_CLIPS_PER_KEYWORD
        for item in source_keywords
    }

    if dataset is None:
        return {}

    for index, row in enumerate(tqdm(dataset, desc=f"Extract {source_label(source)}")):
        scanned_rows += 1
        if scanned_rows > MAX_ROWS_PER_STREAMED_SOURCE:
            break

        candidates = [
            item for item in source_keywords
            if counts[item["model_name"]] < per_keyword_limit[item["model_name"]]
        ]
        if not candidates:
            break

        text = row_text(row, source) if USE_REFERENCE_TRANSCRIPTS_FIRST else ""
        matched = [
            item for item in candidates
            if text and phrase_present(text, item["phrase"])
        ]

        if not matched and RUN_ASR_ON_REFERENCE_MISMATCH and asr_rows < MAX_ASR_ROWS_PER_SOURCE:
            try:
                text = transcribe_audio(row[source.audio_field], source.language)
                asr_rows += 1
                matched = [
                    item for item in candidates
                    if phrase_present(text, item["phrase"])
                ]
            except Exception as exc:
                if asr_rows < 5:
                    print(f"ASR failed on {source_label(source)} row {index}: {exc}")

        for item in matched:
            save_real_keyword_clip(row, source, item["model_name"], index)
            counts[item["model_name"]] += 1

    print(
        f"{source_label(source)} scanned={scanned_rows} asr_rows={asr_rows} "
        f"matches={dict(counts)}"
    )
    return dict(counts)


real_clip_summary = {}
for source in SPEECH_SOURCES:
    dataset = loaded_speech_sources.get(source_label(source))
    real_clip_summary[source_label(source)] = extract_real_clips_from_source(source, dataset)

print(json.dumps(real_clip_summary, indent=2))

## Cell D - Generate Diverse Synthetic TTS Clips

Edge TTS is tried first for multiple voices per phrase. If the Edge websocket is blocked by Colab or Microsoft returns 403, the cell falls back to Hugging Face MMS TTS for English and Tagalog, then saves adult, teenager, child, and elder pitch/rate variants for every phrase.


In [ ]:
import edge_tts
from transformers import AutoModelForTextToWaveform, AutoTokenizer

ENGLISH_TTS_VOICES = {
    "adult_female_aria": "en-US-AriaNeural",
    "adult_male_guy": "en-US-GuyNeural",
    "child_ana": "en-US-AnaNeural",
    "adult_female_jenny": "en-US-JennyNeural",
}

TAGALOG_TTS_VOICES = {
    "adult_male_angelo": "fil-PH-AngeloNeural",
    "adult_female_blessica": "fil-PH-BlessicaNeural",
}

MMS_TTS_MODELS = {
    "en": "facebook/mms-tts-eng",
    "fil": "facebook/mms-tts-tgl",
}
MMS_TTS_CACHE = {}

AGE_VARIANTS = {
    "adult": {"pitch_steps": 0, "rate": 1.00},
    "teenager": {"pitch_steps": 2, "rate": 1.03},
    "child": {"pitch_steps": 6, "rate": 1.08},
    "older": {"pitch_steps": -3, "rate": 0.92},
}


async def available_edge_voices() -> set[str]:
    try:
        voices = await edge_tts.list_voices()
        return {voice["ShortName"] for voice in voices}
    except Exception as exc:
        print(f"Edge TTS voice lookup failed; using MMS fallback ({type(exc).__name__}: {exc})")
        return set()


def tts_text_variants(phrase: str, language: str) -> list[str]:
    phrase = phrase.strip()
    if language == "fil":
        variants = [
            phrase,
            f"{phrase}!",
            f"{phrase} po",
            f"{phrase} ngayon",
            f"{phrase} dito",
            f"paki {phrase}",
            f"{phrase}, please",
            f"{phrase} ako",
            f"{phrase} na",
            f"{phrase}!",
        ]
    else:
        variants = [
            phrase,
            f"{phrase}!",
            f"{phrase} please",
            f"{phrase} now",
            f"{phrase} over here",
            f"please {phrase}",
            f"{phrase}, please",
            f"{phrase}!",
            f"{phrase} right now",
            f"{phrase}",
        ]
    return variants[:10]


async def synthesize_edge_mp3(text: str, voice: str, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(str(output_path))


def get_mms_tts(language: str):
    if language not in MMS_TTS_MODELS:
        raise ValueError(f"No MMS TTS fallback configured for language {language!r}")
    if language not in MMS_TTS_CACHE:
        model_id = MMS_TTS_MODELS[language]
        token_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
        tokenizer = AutoTokenizer.from_pretrained(model_id, **token_kwargs)
        model = AutoModelForTextToWaveform.from_pretrained(model_id, **token_kwargs)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        MMS_TTS_CACHE[language] = {
            "model_id": model_id,
            "tokenizer": tokenizer,
            "model": model,
            "device": device,
            "sample_rate": int(getattr(model.config, "sampling_rate", SAMPLE_RATE)),
        }
        print(f"Loaded MMS TTS fallback {model_id}")
    return MMS_TTS_CACHE[language]


def synthesize_mms_wav(text: str, language: str, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tts = get_mms_tts(language)
    inputs = tts["tokenizer"](text, return_tensors="pt")
    inputs = {key: value.to(tts["device"]) for key, value in inputs.items()}
    with torch.inference_mode():
        waveform = tts["model"](**inputs).waveform
    audio = waveform.squeeze().detach().cpu().numpy().astype(np.float32)
    write_wav_16k(output_path, audio, tts["sample_rate"])


def save_age_variants(source_path: Path, output_dir: Path, base_name: str) -> dict[str, Path]:
    audio, sr = librosa.load(str(source_path), sr=SAMPLE_RATE, mono=True)
    outputs = {}
    for age, settings in AGE_VARIANTS.items():
        variant = audio.astype(np.float32)
        if settings["pitch_steps"]:
            variant = librosa.effects.pitch_shift(
                y=variant,
                sr=SAMPLE_RATE,
                n_steps=float(settings["pitch_steps"]),
            )
        if abs(float(settings["rate"]) - 1.0) > 1e-3:
            variant = librosa.effects.time_stretch(variant, rate=float(settings["rate"]))
        output_path = output_dir / f"{base_name}_{age}.wav"
        sf.write(str(output_path), np.clip(variant, -1.0, 1.0), SAMPLE_RATE, subtype="PCM_16")
        outputs[age] = output_path
    return outputs


async def generate_tts_clips() -> dict[str, dict[str, int]]:
    available = await available_edge_voices()
    summary = defaultdict(lambda: defaultdict(int))
    work_dir = WORK_DIR / "tts_work"
    edge_tts_disabled = False
    mms_notice_printed = False

    for item in KEYWORD_SPECS:
        phrase = item["phrase"]
        model_name = item["model_name"]
        language = item["language"]
        voices = TAGALOG_TTS_VOICES if language == "fil" else ENGLISH_TTS_VOICES
        selected_voices = {
            label: voice for label, voice in voices.items()
            if voice in available
        }
        if edge_tts_disabled:
            selected_voices = {}
        if language == "fil" and not selected_voices and available and not edge_tts_disabled:
            print("Tagalog Edge voices unavailable; falling back to English voices for", phrase)
            selected_voices = {
                label: voice for label, voice in ENGLISH_TTS_VOICES.items()
                if voice in available
            }
        if not selected_voices and not mms_notice_printed:
            print("Using Hugging Face MMS TTS fallback for synthetic clips.")
            mms_notice_printed = True

        output_dir = TTS_CLIPS_DIR / language / model_name
        output_dir.mkdir(parents=True, exist_ok=True)
        for variant_index, text in enumerate(tts_text_variants(phrase, language)):
            if selected_voices and not edge_tts_disabled:
                for voice_label, voice in selected_voices.items():
                    base_name = f"{model_name}_{voice_label}_v{variant_index:02d}"
                    mp3_path = work_dir / language / model_name / f"{base_name}.mp3"
                    try:
                        if not mp3_path.exists():
                            await synthesize_edge_mp3(text, voice, mp3_path)
                        outputs = save_age_variants(mp3_path, output_dir, base_name)
                        for age in outputs:
                            summary[model_name][age] += 1
                    except Exception as exc:
                        edge_tts_disabled = True
                        if mp3_path.exists() and mp3_path.stat().st_size == 0:
                            mp3_path.unlink()
                        print(f"Edge TTS failed ({type(exc).__name__}: {exc}). Switching to MMS fallback for the rest of Cell D.")
                        if not mms_notice_printed:
                            print("Using Hugging Face MMS TTS fallback for synthetic clips.")
                            mms_notice_printed = True
                        break
            if edge_tts_disabled or not selected_voices:
                base_name = f"{model_name}_mms_{language}_v{variant_index:02d}"
                wav_path = work_dir / language / model_name / f"{base_name}.wav"
                if not wav_path.exists():
                    synthesize_mms_wav(text, language, wav_path)
                outputs = save_age_variants(wav_path, output_dir, base_name)
                for age in outputs:
                    summary[model_name][age] += 1

    return {key: dict(value) for key, value in summary.items()}


tts_summary = await generate_tts_clips()
print(json.dumps(tts_summary, indent=2))

## Cell E - Merge, Split, And Balance Positive Clips

All real and TTS clips are copied into `positive_clips/<model_name>/{train,val}` and held-out real/TTS clips go to `heldout_clips/<model_name>`. If a phrase has fewer than 500 clips, light augmentation creates more positives without changing the target phrase.


In [ ]:
BALANCE_AUGMENTER = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.008, p=0.8),
    Gain(min_gain_db=-3.0, max_gain_db=3.0, p=0.8),
    PitchShift(min_semitones=-1.0, max_semitones=1.0, p=0.5),
    TimeStretch(min_rate=0.92, max_rate=1.08, p=0.4),
])


def file_sha1(path: Path) -> str:
    digest = hashlib.sha1()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_clip(path: Path) -> np.ndarray:
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    audio = _to_mono(audio)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    return np.asarray(audio, dtype=np.float32)


def copy_unique_clip(source: Path, destination: Path, seen_hashes: set[str]) -> bool:
    digest = file_sha1(source)
    if digest in seen_hashes:
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    seen_hashes.add(digest)
    return True


def candidate_positive_files(item: dict) -> list[tuple[Path, str, str]]:
    language = item["language"]
    model_name = item["model_name"]
    candidates = []
    for path in sorted((REAL_CLIPS_DIR / language / model_name).glob("*.wav")):
        parts = path.stem.split("_")
        age = next((part for part in parts if part in {"child", "teenager", "adult", "older"}), "unknown")
        candidates.append((path, "real", age))
    for path in sorted((TTS_CLIPS_DIR / language / model_name).glob("*.wav")):
        age = next((age for age in AGE_VARIANTS if path.stem.endswith(f"_{age}")), "adult")
        candidates.append((path, "tts", age))
    return candidates


def balance_training_clips(train_dir: Path, target_count: int) -> int:
    wavs = sorted(train_dir.glob("*.wav"))
    if not wavs:
        return 0
    created = 0
    next_index = 0
    with tqdm(total=max(0, target_count - len(wavs)), desc=f"Balance {train_dir.parent.name}") as bar:
        while len(wavs) + created < target_count:
            source = wavs[next_index % len(wavs)]
            audio = load_clip(source)
            augmented = BALANCE_AUGMENTER(samples=audio, sample_rate=SAMPLE_RATE)
            output = train_dir / f"aug_{created:05d}_{source.name}"
            sf.write(str(output), np.clip(augmented, -1.0, 1.0), SAMPLE_RATE, subtype="PCM_16")
            created += 1
            next_index += 1
            bar.update(1)
    return created


def merge_and_balance_keyword(item: dict) -> dict[str, object]:
    model_name = item["model_name"]
    language = item["language"]
    all_candidates = candidate_positive_files(item)
    rng = random.Random(f"{RANDOM_SEED}:{model_name}")
    rng.shuffle(all_candidates)

    train_dir = POSITIVE_CLIPS_DIR / model_name / "train"
    val_dir = POSITIVE_CLIPS_DIR / model_name / "val"
    heldout_dir = HELDOUT_CLIPS_DIR / model_name
    for directory in [train_dir, val_dir, heldout_dir]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

    seen = set()
    summary = Counter()
    heldout_count = max(1, int(len(all_candidates) * HELDOUT_FRACTION)) if all_candidates else 0

    for index, (source, source_kind, age) in enumerate(all_candidates):
        split = "heldout" if index < heldout_count else ("val" if index % 7 == 0 else "train")
        destination_dir = heldout_dir if split == "heldout" else val_dir if split == "val" else train_dir
        destination = destination_dir / f"{source_kind}_{age}_{source.name}"
        if copy_unique_clip(source, destination, seen):
            summary[f"{split}_{source_kind}_{age}"] += 1

    augmented = balance_training_clips(train_dir, MIN_POSITIVE_CLIPS_PER_KEYWORD)
    summary["train_augmented"] = augmented
    summary["train_total"] = len(list(train_dir.glob("*.wav")))
    summary["val_total"] = len(list(val_dir.glob("*.wav")))
    summary["heldout_total"] = len(list(heldout_dir.glob("*.wav")))
    summary["language"] = language
    return dict(summary)


positive_clip_summary = {
    item["model_name"]: merge_and_balance_keyword(item)
    for item in KEYWORD_SPECS
}

summary_df = pd.DataFrame.from_dict(positive_clip_summary, orient="index").fillna(0)
display(summary_df)
summary_df.to_csv(WORK_DIR / "positive_clip_summary.csv")
print(f"Wrote summary: {WORK_DIR / 'positive_clip_summary.csv'}")

## Cell F - Prepare openWakeWord And Train Models

The Pi consumes ONNX only. This cell patches openWakeWord's TFLite conversion path so training still succeeds on Colab stacks where TensorFlow export helpers are fragile. By default, training uses the real/TTS clips prepared above and skips openWakeWord's Piper clip generation stage.


In [ ]:
def run_command(args, cwd=None, timeout=None):
    printable = " ".join(str(arg) for arg in args)
    print("+", printable)
    process = subprocess.Popen(
        [str(arg) for arg in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            tail.append(line.rstrip())
            tail = tail[-80:]
        return_code = process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        process.kill()
        raise
    if return_code != 0:
        print("Command failed. Last output lines:")
        print("\n".join(tail[-80:]))
        raise subprocess.CalledProcessError(return_code, [str(arg) for arg in args], output="\n".join(tail))


def pip_install(*args):
    run_command([sys.executable, "-m", "pip", "install", "-q", *args])


def ensure_openwakeword_runtime_deps() -> None:
    try:
        import onnxruntime  # noqa: F401
    except ImportError:
        print("Installing missing openWakeWord runtime dependency: onnxruntime")
        pip_install("onnxruntime")


def patch_torchaudio_info() -> None:
    import torchaudio

    ta_path = Path(inspect.getfile(torchaudio))
    ta_text = ta_path.read_text(encoding="utf-8")

    patch_marker = "# Project Pi torchaudio.info compatibility patch"

    if patch_marker not in ta_text:
        ta_path.write_text(
            ta_text
            + r'''

# Project Pi torchaudio.info compatibility patch
if "info" not in globals():
    class AudioMetaData:
        def __init__(
            self,
            sample_rate,
            num_frames,
            num_channels=1,
            bits_per_sample=0,
            encoding="UNKNOWN",
        ):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding

    def info(uri, format=None, buffer_size=4096, backend=None):
        del format, buffer_size, backend
        import soundfile as _sf

        metadata = _sf.info(str(uri))
        return AudioMetaData(
            sample_rate=int(metadata.samplerate),
            num_frames=int(metadata.frames),
            num_channels=int(metadata.channels),
            bits_per_sample=0,
            encoding=str(metadata.format or "UNKNOWN"),
        )
''',
            encoding="utf-8",
        )
        print(f"Patched torchaudio.info fallback: {ta_path}")
    else:
        print("torchaudio.info patch already present")

    importlib.reload(torchaudio)
    assert hasattr(torchaudio, "info"), "torchaudio.info patch failed"
    print("torchaudio.info is ready")


def clone_or_update_repo(url: str, destination: Path) -> None:
    if destination.exists():
        run_command(["git", "-C", destination, "pull", "--ff-only"])
    else:
        run_command(["git", "clone", url, destination])


def install_openwakeword_training_stack():
    clone_or_update_repo("https://github.com/rhasspy/piper-sample-generator", PIPER_ROOT)
    (PIPER_ROOT / "models").mkdir(parents=True, exist_ok=True)
    piper_voice = PIPER_ROOT / "models" / "en_US-libritts_r-medium.pt"
    if not piper_voice.exists():
        run_command([
            "wget",
            "-O",
            piper_voice,
            "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",
        ])

    clone_or_update_repo("https://github.com/dscripka/openwakeword", OPENWAKEWORD_ROOT)

    pip_install("--no-deps", "-e", str(PIPER_ROOT))
    pip_install("--no-deps", "-e", str(OPENWAKEWORD_ROOT))
    pip_install(
        "-c",
        "project_pi_constraints.txt",
        "onnxruntime",
        "webrtcvad",
        "mutagen==1.47.0",
        "torchinfo==1.8.0",
        "torchmetrics==1.2.0",
        "speechbrain==0.5.14",
        "torch-audiomentations==0.12.0",
        "acoustics==0.2.6",
        "pronouncing==0.2.0",
        "deep-phonemizer==0.0.19",
        "soundfile>=0.12.1,<0.14",
        "pyyaml",
    )


def patch_piper_and_openwakeword():
    shim_path = PIPER_ROOT / "generate_samples.py"
    shim_path.write_text(
        """from pathlib import Path
from piper_sample_generator.__main__ import generate_samples as _generate_samples

_DEFAULT_MODEL = Path(__file__).parent / "models" / "en_US-libritts_r-medium.pt"

def generate_samples(*args, model=None, **kwargs):
    return _generate_samples(*args, model=model or _DEFAULT_MODEL, **kwargs)
""",
        encoding="utf-8",
    )

    resource_dir = OPENWAKEWORD_ROOT / "openwakeword" / "resources" / "models"
    resource_dir.mkdir(parents=True, exist_ok=True)
    model_urls = {
        "embedding_model.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
        "embedding_model.tflite": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite",
        "melspectrogram.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
        "melspectrogram.tflite": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite",
    }
    for filename, url in model_urls.items():
        output_path = resource_dir / filename
        if not output_path.exists():
            run_command(["wget", "-nc", url, "-O", output_path])

    import dp.model.model as dp_model
    dp_path = Path(inspect.getfile(dp_model))
    text = dp_path.read_text(encoding="utf-8")
    old = "checkpoint = torch.load(checkpoint_path, map_location=device)"
    new = "checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)"
    if old in text and new not in text:
        dp_path.write_text(text.replace(old, new), encoding="utf-8")
        print(f"Patched deep-phonemizer: {dp_path}")

    train_path = OPENWAKEWORD_ROOT / "openwakeword" / "train.py"
    train_text = train_path.read_text(encoding="utf-8")
    lazy_piper_marker = "# Project Pi lazy Piper import"
    if lazy_piper_marker not in train_text:
        train_text = train_text.replace(
            "    # imports Piper for synthetic sample generation\n"
            "    sys.path.insert(0, os.path.abspath(config[\"piper_sample_generator_path\"]))\n"
            "    from generate_samples import generate_samples\n",
            "    # Project Pi lazy Piper import\n"
            "    generate_samples = None\n"
            "    if args.generate_clips is True:\n"
            "        sys.path.insert(0, os.path.abspath(config[\"piper_sample_generator_path\"]))\n"
            "        from generate_samples import generate_samples\n",
            1,
        )
    marker = "# Project Pi skip TFLite conversion"
    if marker not in train_text:
        train_text = train_text.replace(
            "def convert_onnx_to_tflite(onnx_model_path, output_path):\n",
            "def convert_onnx_to_tflite(onnx_model_path, output_path):\n"
            f"    # {marker}\n"
            "    print(f'Skipping TFLite conversion; ONNX model remains at {onnx_model_path}')\n"
            "    return None\n",
            1,
        )
        train_path.write_text(train_text, encoding="utf-8")
        print("Patched openWakeWord train.py to keep ONNX only")


install_openwakeword_training_stack()
ensure_openwakeword_runtime_deps()
patch_piper_and_openwakeword()
patch_torchaudio_info()
print("openWakeWord training stack ready")

In [ ]:
TRAINING_NOTEBOOK_REVISION = "2026-08-19-torchaudio-info-v1"
MIN_OPENWAKEWORD_TEST_CLIPS = 50
NEGATIVE_CLIPS_PER_SPLIT = 500 if not FINAL_TRAINING else 5000


def wav_dir_has_audio(path: Path) -> bool:
    return Path(path).exists() and any(Path(path).glob("*.wav"))


def build_training_overrides(final_training: bool) -> dict:
    background_candidates = [
        (AUGMENTATION_DIR / "audioset_16k", 2),
        (AUGMENTATION_DIR / "fma", 1),
        (AUGMENTATION_DIR / "synthetic_noise", 1),
    ]
    background_pairs = [
        (path, rate) for path, rate in background_candidates
        if wav_dir_has_audio(path)
    ]
    rir_paths = [
        str(path) for path in [AUGMENTATION_DIR / "mit_rirs"]
        if wav_dir_has_audio(path)
    ]

    return {
        "output_dir": str(MODEL_ROOT),
        "piper_sample_generator_path": str(PIPER_ROOT),
        "n_samples": 50000 if final_training else 8000,
        "n_samples_val": 5000 if final_training else 1000,
        "steps": 50000 if final_training else 12000,
        "target_accuracy": 0.80 if final_training else 0.60,
        "target_recall": 0.60 if final_training else 0.25,
        "target_false_positives_per_hour": 0.05 if final_training else 0.20,
        "max_negative_weight": 2500 if final_training else 1000,
        "augmentation_rounds": 2 if final_training else 1,
        "background_paths": [str(path) for path, _ in background_pairs],
        "background_paths_duplication_rate": [rate for _, rate in background_pairs],
        "rir_paths": rir_paths,
        "false_positive_validation_data_path": "validation_set_features.npy",
        "feature_data_files": {
            "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
        },
    }


def write_training_config(item: dict, final_training: bool = FINAL_TRAINING) -> Path:
    config_path = OPENWAKEWORD_ROOT / "examples" / "custom_model.yml"
    config = __import__("yaml").safe_load(config_path.read_text(encoding="utf-8"))
    config.update(build_training_overrides(final_training))
    config["target_phrase"] = [item["phrase"]]
    config["model_name"] = item["model_name"]
    config["custom_negative_phrases"] = []

    train_dir = POSITIVE_CLIPS_DIR / item["model_name"] / "train"
    val_dir = POSITIVE_CLIPS_DIR / item["model_name"] / "val"
    if "positive_train_dir" in config:
        config["positive_train_dir"] = str(train_dir)
    if "positive_val_dir" in config:
        config["positive_val_dir"] = str(val_dir)

    output_path = WORK_DIR / "my_model.yaml"
    output_path.write_text(
        __import__("yaml").safe_dump(config, sort_keys=False),
        encoding="utf-8",
    )
    print(
        f"Configured {item['model_name']}: {item['phrase']} "
        f"final_training={final_training}"
    )
    return output_path


def model_dir(model_name: str) -> Path:
    return MODEL_ROOT / model_name


def reset_model_outputs(model_name: str) -> None:
    for path in [
        MODEL_ROOT / model_name,
        MODEL_ROOT / f"{model_name}.onnx",
        MODEL_ROOT / f"{model_name}.onnx.data",
        MODEL_ROOT / f"{model_name}.tflite",
        MODEL_ROOT / f"{model_name}.pt",
    ]:
        if path.is_dir():
            shutil.rmtree(path)
        elif path.exists():
            path.unlink()
    model_dir(model_name).mkdir(parents=True, exist_ok=True)


def find_positive_dirs(root: Path) -> tuple[Path, Path]:
    positive_dirs = [
        path for path in root.rglob("*")
        if path.is_dir() and "positive" in path.name.lower()
    ]
    train_candidates = [
        path for path in positive_dirs
        if "train" in path.name.lower()
    ]
    val_candidates = [
        path for path in positive_dirs
        if any(token in path.name.lower() for token in ["val", "test"])
    ]
    train_dir = (train_candidates or positive_dirs or [root / "positive_train"])[0]
    val_dir = (val_candidates or [root / "positive_test"])[0]
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    return train_dir, val_dir


def inject_positive_clips(model_name: str) -> dict[str, int]:
    generated_train, generated_val = find_positive_dirs(model_dir(model_name))
    source_train = POSITIVE_CLIPS_DIR / model_name / "train"
    source_val = POSITIVE_CLIPS_DIR / model_name / "val"
    copied = Counter()
    for source_dir, target_dir, split in [
        (source_train, generated_train, "train"),
        (source_val, generated_val, "val"),
    ]:
        if not source_dir.exists():
            continue
        existing = {file_sha1(path) for path in target_dir.glob("*.wav")}
        for source_path in sorted(source_dir.glob("*.wav")):
            target_path = target_dir / f"project_pi_{source_path.name}"
            if copy_unique_clip(source_path, target_path, existing):
                copied[split] += 1
    print(f"Injected positives for {model_name}: {dict(copied)}")
    return dict(copied)


def openwakeword_clip_dirs(model_name: str) -> dict[str, Path]:
    root = model_dir(model_name)
    dirs = {
        "positive_train": root / "positive_train",
        "positive_test": root / "positive_test",
        "negative_train": root / "negative_train",
        "negative_test": root / "negative_test",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def copy_positive_split(source_dir: Path, target_dir: Path, prefix: str) -> int:
    seen = {file_sha1(path) for path in target_dir.glob("*.wav")}
    copied = 0
    for source_path in sorted(source_dir.glob("*.wav")):
        target_path = target_dir / f"{prefix}_{source_path.name}"
        if copy_unique_clip(source_path, target_path, seen):
            copied += 1
    return copied


def seed_positive_training_dirs(model_name: str, dirs: dict[str, Path]) -> dict[str, int]:
    source_train = POSITIVE_CLIPS_DIR / model_name / "train"
    source_val = POSITIVE_CLIPS_DIR / model_name / "val"
    if not wav_dir_has_audio(source_train):
        raise FileNotFoundError(
            f"No balanced positive clips found in {source_train}. Run Cell E before Cell F."
        )

    train_count = copy_positive_split(source_train, dirs["positive_train"], "project_pi_train")
    test_count = copy_positive_split(source_val, dirs["positive_test"], "project_pi_val") if source_val.exists() else 0

    if count_wavs(dirs["positive_test"]) < MIN_OPENWAKEWORD_TEST_CLIPS:
        current = count_wavs(dirs["positive_test"])
        for source_path in sorted(source_train.glob("*.wav")):
            if count_wavs(dirs["positive_test"]) >= MIN_OPENWAKEWORD_TEST_CLIPS:
                break
            target_path = dirs["positive_test"] / f"project_pi_val_extra_{current:04d}_{source_path.name}"
            shutil.copy2(source_path, target_path)
            current += 1
            test_count += 1

    return {
        "positive_train": count_wavs(dirs["positive_train"]),
        "positive_test": count_wavs(dirs["positive_test"]),
        "positive_train_copied": train_count,
        "positive_test_copied": test_count,
    }


def negative_source_files() -> list[Path]:
    candidates = []
    for root in [
        AUGMENTATION_DIR / "audioset_16k",
        AUGMENTATION_DIR / "fma",
        AUGMENTATION_DIR / "synthetic_noise",
    ]:
        candidates.extend(sorted(root.glob("*.wav")))
    if not candidates:
        create_synthetic_backgrounds(max(80, NEGATIVE_CLIPS_PER_SPLIT // 2))
        candidates.extend(sorted((AUGMENTATION_DIR / "synthetic_noise").glob("*.wav")))
    if not candidates:
        raise FileNotFoundError("No negative/background WAVs found. Run Cell B2 before Cell F.")
    return candidates


def seed_negative_training_dirs(model_name: str, dirs: dict[str, Path]) -> dict[str, int]:
    sources = negative_source_files()
    rng = random.Random(f"{RANDOM_SEED}:negative:{model_name}")
    rng.shuffle(sources)
    summary = {}
    split_targets = {
        "negative_train": NEGATIVE_CLIPS_PER_SPLIT,
        "negative_test": max(MIN_OPENWAKEWORD_TEST_CLIPS, NEGATIVE_CLIPS_PER_SPLIT // 5),
    }
    for split, target_count in split_targets.items():
        target_dir = dirs[split]
        existing = count_wavs(target_dir)
        index = 0
        while existing < target_count:
            source_path = sources[index % len(sources)]
            target_path = target_dir / f"project_pi_bg_{existing:05d}_{source_path.name}"
            shutil.copy2(source_path, target_path)
            existing += 1
            index += 1
        summary[split] = count_wavs(target_dir)
    return summary


def prepare_openwakeword_training_dirs(model_name: str) -> dict[str, int]:
    dirs = openwakeword_clip_dirs(model_name)
    summary = {}
    summary.update(seed_positive_training_dirs(model_name, dirs))
    summary.update(seed_negative_training_dirs(model_name, dirs))
    print(f"Prepared openWakeWord clips for {model_name}: {summary}")
    return summary


def clear_feature_cache(model_name: str) -> None:
    for file in model_dir(model_name).glob("*features*.npy"):
        file.unlink()


def run_stage(stage: str, config_path: Path) -> None:
    run_command([
        sys.executable,
        OPENWAKEWORD_ROOT / "openwakeword" / "train.py",
        "--training_config",
        config_path,
        f"--{stage}",
    ])


def verify_exported_model(model_name: str) -> list[Path]:
    exported = [
        MODEL_ROOT / f"{model_name}.onnx",
        MODEL_ROOT / f"{model_name}.onnx.data",
    ]
    missing = [str(path) for path in exported if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing exported ONNX artifact(s): {missing}")
    for path in exported:
        print(f"Exported {path} ({path.stat().st_size / 1024:.1f} KB)")
    return exported


def train_one_keyword(item: dict, final_training: bool = FINAL_TRAINING) -> list[Path]:
    ensure_openwakeword_runtime_deps()
    patch_torchaudio_info()
    model_name = item["model_name"]
    config_path = write_training_config(item, final_training)
    reset_model_outputs(model_name)
    prepare_openwakeword_training_dirs(model_name)
    print("Skipped openWakeWord Piper generation; using prepared real/TTS positives and background negatives.")
    clear_feature_cache(model_name)
    run_stage("augment_clips", config_path)
    run_stage("train_model", config_path)
    return verify_exported_model(model_name)


def train_all_keywords(final_training: bool = FINAL_TRAINING) -> dict[str, list[str]]:
    trained = {}
    for item in KEYWORD_SPECS:
        artifacts = train_one_keyword(item, final_training=final_training)
        trained[item["model_name"]] = [str(path) for path in artifacts]
    return trained


print("Training helpers ready")
print("Profile:", RUNTIME_PROFILE, build_training_overrides(FINAL_TRAINING))

In [ ]:
# Run training. For a quick validation, set TRAIN_KEYWORDS near the top to a short list before rerunning.
if globals().get("TRAINING_NOTEBOOK_REVISION") != "2026-08-19-torchaudio-info-v1":
    raise RuntimeError(
        "Stale Cell F helper definitions detected. Refresh/reopen the GitHub notebook, "
        "then rerun Cell F's setup/helper cells before this training cell."
    )
print("Training notebook revision:", TRAINING_NOTEBOOK_REVISION)
trained_artifacts = train_all_keywords(final_training=FINAL_TRAINING)
print(json.dumps(trained_artifacts, indent=2))

## Cell G - Evaluate Held-Out Positives And Background Negatives

This is a fast clip-level check, not a full deployment validation. Use it to compare thresholds and catch broken models before copying them to the Pi.


In [ ]:
def wav_to_int16_chunks(path: Path, chunk_size: int = 1280):
    audio = load_clip(path)
    pcm = np.clip(audio, -1.0, 1.0)
    pcm = (pcm * 32767).astype(np.int16)
    for start in range(0, len(pcm), chunk_size):
        chunk = pcm[start : start + chunk_size]
        if chunk.size < chunk_size:
            padded = np.zeros(chunk_size, dtype=np.int16)
            padded[: chunk.size] = chunk
            chunk = padded
        yield chunk


def score_wav_with_model(model, path: Path) -> float:
    max_score = 0.0
    if hasattr(model, "reset"):
        model.reset()
    for chunk in wav_to_int16_chunks(path):
        prediction = model.predict(chunk)
        for name, value in prediction.items():
            if name == "vad":
                continue
            try:
                score = float(np.asarray(value).reshape(-1)[-1])
            except Exception:
                score = 0.0
            max_score = max(max_score, score)
    return max_score


def negative_evaluation_files() -> list[Path]:
    candidates = []
    for root in [
        AUGMENTATION_DIR / "audioset_16k",
        AUGMENTATION_DIR / "fma",
        AUGMENTATION_DIR / "synthetic_noise",
    ]:
        candidates.extend(sorted(root.glob("*.wav")))
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(candidates)
    return candidates[:EVALUATION_MAX_NEGATIVES]


def evaluate_one_model(item: dict, threshold: float = 0.5) -> dict[str, object]:
    from openwakeword.model import Model

    model_name = item["model_name"]
    model_path = MODEL_ROOT / f"{model_name}.onnx"
    if not model_path.exists():
        return {
            "model_name": model_name,
            "phrase": item["phrase"],
            "error": f"missing {model_path}",
        }

    model = Model(
        wakeword_models=[str(model_path)],
        inference_framework="onnx",
        enable_speex_noise_suppression=False,
    )

    positives = sorted((HELDOUT_CLIPS_DIR / model_name).glob("*.wav"))[:EVALUATION_MAX_POSITIVES]
    negatives = negative_evaluation_files()
    positive_scores = [score_wav_with_model(model, path) for path in tqdm(positives, desc=f"{model_name} positives")]
    negative_scores = [score_wav_with_model(model, path) for path in tqdm(negatives, desc=f"{model_name} negatives")]

    tp = sum(score >= threshold for score in positive_scores)
    fn = len(positive_scores) - tp
    fp = sum(score >= threshold for score in negative_scores)
    tn = len(negative_scores) - fp
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    false_accept_rate = fp / max(len(negative_scores), 1)
    return {
        "model_name": model_name,
        "phrase": item["phrase"],
        "threshold": threshold,
        "positives": len(positive_scores),
        "negatives": len(negative_scores),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "false_accept_rate": round(false_accept_rate, 4),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "max_positive_score": round(max(positive_scores or [0.0]), 4),
        "max_negative_score": round(max(negative_scores or [0.0]), 4),
    }


evaluation_rows = [
    evaluate_one_model(item, threshold=0.5)
    for item in KEYWORD_SPECS
]
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_path = WORK_DIR / "evaluation_metrics.csv"
evaluation_df.to_csv(evaluation_path, index=False)
print(f"Wrote evaluation metrics: {evaluation_path}")

## Cell H - Download ONNX Models

Downloads individual ONNX artifacts and creates one zip file containing every exported model and companion data file. Copy both files for each phrase to `raspberry_pi/kws/openwakeword_models/` on the Pi.


In [ ]:
def package_onnx_models() -> Path:
    zip_path = MODEL_ROOT / "project_pi_openwakeword_onnx_models.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in KEYWORD_SPECS:
            model_name = item["model_name"]
            for suffix in [".onnx", ".onnx.data"]:
                artifact = MODEL_ROOT / f"{model_name}{suffix}"
                if artifact.exists():
                    archive.write(artifact, artifact.name)
                else:
                    print(f"Missing, not zipped: {artifact}")
    print(f"Created {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
    return zip_path


zip_path = package_onnx_models()

try:
    from google.colab import files

    files.download(str(zip_path))
    DOWNLOAD_INDIVIDUAL_FILES = False
    if DOWNLOAD_INDIVIDUAL_FILES:
        for item in KEYWORD_SPECS:
            model_name = item["model_name"]
            for suffix in [".onnx", ".onnx.data"]:
                artifact = MODEL_ROOT / f"{model_name}{suffix}"
                if artifact.exists():
                    files.download(str(artifact))
except Exception as exc:
    print(f"Colab download helper unavailable: {exc}")
    print(f"Download manually from: {zip_path}")

## Pi Integration

After downloading, copy every `.onnx` file and its `.onnx.data` companion to:

```text
/home/thesis/Project_Pi/raspberry_pi/kws/openwakeword_models/
```

Then restart the Pi keyword service:

```bash
sudo systemctl restart kws-alert.service
sudo journalctl -u kws-alert.service -f
```

The existing `audio_preprocessor.py` discovers all `.onnx` files automatically and maps filenames like `help_me.onnx` to the phrase `help me`.
